In [ ]:
# RAG (Retrieval Augmented Generation) based on a dataset of 800,000 scraped Amazon products
# For our 2nd agent, we will be asking OpenAI to estimate the price of one of our deals 



#---------(1) --------------Below is simple RAG and will combine with fronter model gpt 5-- to see the error rate
# So basically it says we have some items and prices, i want to do similar items lookup 
import os
import logging
from dotenv import load_dotenv
from huggingface_hub import login
import numpy as np
import re
from sentence_transformers import SentenceTransformer
import chromadb
from sklearn.manifold import TSNE
import plotly.graph_objects as go
from litellm import completion
from tqdm.notebook import tqdm
from agents.evaluator import evaluate
from agents.items import Item

# environment
load_dotenv(override=True)
DB = "products_vectorstore"

hf_token = os.environ['HF_TOKEN']
login(token=hf_token, add_to_git_credential=False)
LITE_MODE = False
username = "allanhadoop"     # "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"
train, val, test = Item.from_hub(dataset)
print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

# We will create a Chroma datastore with 400,000 products from our training dataset.
client = chromadb.PersistentClient(path=DB)

encoder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
# Pass in a list of texts, get back a numpy array of vectors
vector = encoder.encode(["A proficient AI engineer who has almost reached the finale of AI Engineering Core Track!"])[0]
print(vector.shape)
vector

# Check if the collection exists; if not, create it . Calculate  vectors for 800,000 scraped products
collection_name = "products"
existing_collection_names = [collection.name for collection in client.list_collections()]
if collection_name not in existing_collection_names:
    collection = client.create_collection(collection_name)
    for i in tqdm(range(0, len(train), 1000)):
        documents = [item.summary for item in train[i: i+1000]]
        vectors = encoder.encode(documents).astype(float).tolist()
        metadatas = [{"category": item.category, "price": item.price} for item in train[i: i+1000]]
        ids = [f"doc_{j}" for j in range(i, i+1000)]
        ids = ids[:len(documents)]
        collection.add(ids=ids, documents=documents, embeddings=vectors, metadatas=metadatas)
collection = client.get_or_create_collection(collection_name)

# It is very fun turning this up to 800_000 and seeing the full dataset visualized,
# but it almost crashes my box every time so do that at your own risk!! 10_000 is safe!
MAXIMUM_DATAPOINTS = 10_000
CATEGORIES = ['Appliances', 'Automotive', 'Cell_Phones_and_Accessories', 'Electronics','Musical_Instruments', 'Office_Products', 'Tools_and_Home_Improvement', 'Toys_and_Games']
COLORS = ['cyan', 'blue', 'brown', 'orange', 'yellow', 'green' , 'purple', 'red']

# Prework
result = collection.get(include=['embeddings', 'documents', 'metadatas'], limit=MAXIMUM_DATAPOINTS)
vectors = np.array(result['embeddings'])
documents = result['documents']
categories = [metadata['category'] for metadata in result['metadatas']]
colors = [COLORS[CATEGORIES.index(c)] for c in categories]
# Let's try a 2D chart
# TSNE stands for t-distributed Stochastic Neighbor Embedding - it's a common technique for reducing dimensionality of data
tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=4, color=colors, opacity=0.7),
    text=[f"Category: {c}<br>Text: {d[:50]}..." for c, d in zip(categories, documents)],
    hoverinfo='text'
)])
fig.update_layout(
    title='2D Chroma Vectorstore Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y'),
    width=1200,
    height=800,
    margin=dict(r=20, b=10, l=10, t=40)
)
fig.show()

# 3d 
tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)
# Create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=2, color=colors, opacity=0.7),
    text=[f"Category: {c}<br>Text: {d[:50]}..." for c, d in zip(categories, documents)],
    hoverinfo='text'
)])
fig.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=1200,
    height=800,
    margin=dict(r=20, b=10, l=10, t=40)
)
fig.show()
test[0]
def vector(item):
    return encoder.encode(item.summary)
def find_similars(item):
    vec = vector(item)
    results = collection.query(query_embeddings=vec.astype(float).tolist(), n_results=5)
    documents = results['documents'][0][:]
    prices = [m['price'] for m in results['metadatas'][0][:]]
    return documents, prices
find_similars(test[0])

# We need to give some context to GPT-5.1 by selecting 5 products with similar descriptions
def make_context(similars, prices):
    message = "For context, here are some other items that might be similar to the item you need to estimate.\n\n"
    for similar, price in zip(similars, prices):
        message += f"Potentially related product:\n{similar}\nPrice is ${price:.2f}\n\n"
    return message

documents, prices = find_similars(test[0])
print(make_context(documents, prices))
def messages_for(item, similars, prices):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}\n\n"
    message += make_context(similars, prices)
    return [{"role": "user", "content": message}]
documents, prices = find_similars(test[0])
print(messages_for(test[0], documents, prices)[0]['content'])

# The function for gpt-5-mini
def gpt_5__1_rag(item):
    documents, prices = find_similars(item)
    response = completion(model="gpt-5.1", messages=messages_for(item, documents, prices), reasoning_effort="none", seed=42)
    return response.choices[0].message.content
test[0].price
gpt_5__1_rag(test[0])
evaluate(gpt_5__1_rag, test)
### ----------With above RAG and GPT 5 - we get error rate of $30.19 -- WHICH IS EVEN BETTER THAN OUR FINE TUNED MODEL 

#-----------(2) ------------Now get our tuned model from MODAL.COM
import modal
Pricer = modal.Cls.from_name("pricer-service", "Pricer")
pricer = Pricer()
def specialist(item):
    return pricer.price.remote(item.summary)
def get_price(reply):
    reply = reply.replace("$", "").replace(",", "")
    match = re.search(r"[-+]?\d*\.\d+|\d+", reply)
    return float(match.group()) if match else 0


#-----------(3) ------------Now get previously built deep neural network model 
# Download the Neural Network weights from Week 6 into this directory
# File is at https://drive.google.com/drive/folders/1uq5C9edPIZ1973dArZiEO-VE13F7m8MK?usp=drive_link

from agents.deep_neural_network import DeepNeuralNetworkInference
runner = DeepNeuralNetworkInference()
runner.setup()
runner.load("deep_neural_network.pth")
def deep_neural_network(item):
    return runner.inference(item.summary)


#---------------------Now combine all of above three models into Ensemble Model below------------------------
def ensemble(item):
    price1 = get_price(gpt_5__1_rag(item))
    price2 = specialist(item)
    price3 = deep_neural_network(item)
    return price1 * 0.8 + price2 * 0.1 + price3 * 0.1


evaluate(ensemble, test)
## With ensemble, we now get error rate of $29.90 which is better than all models 





